# Proyek Analisis Sentimen Google Play

Notebook ini melatih model analisis sentimen tiga kelas (`negatif`, `netral`, `positif`) menggunakan dataset hasil scraping mandiri dari ulasan Google Play. Dataset dilabeli dengan fitur leksikon sentimen bahasa Indonesia pada teks ulasan, kemudian diuji dengan tiga skema pelatihan berbeda.

## 1. Import dan Load Dataset

In [1]:
from pathlib import Path

from train_sentiment import load_dataset, label_distribution

dataset_path = Path('google_play_reviews_sentiment.csv')
examples = load_dataset(dataset_path)

print(f'Dataset rows: {len(examples)}')
print(f'Label distribution: {label_distribution(examples)}')

Dataset rows: 12000
Label distribution: {'negatif': 4839, 'netral': 3991, 'positif': 3170}


## 2. Training Tiga Skema Model

Tiga skema dibedakan dari metode ekstraksi fitur, rentang n-gram, rasio pembagian data, dan nilai smoothing model Naive Bayes.

In [2]:
from train_sentiment import default_experiments, train_experiment

results = []
for experiment in default_experiments():
    result = train_experiment(examples, experiment)
    results.append(result)

    print(experiment.name)
    print(f"Train size: {result['train_size']}")
    print(f"Test size: {result['test_size']}")
    print(f"Vocabulary size: {result['vocabulary_size']}")
    print(f"Training accuracy: {result['train_accuracy']:.4f}")
    print(f"Testing accuracy: {result['test_accuracy']:.4f}")
    print(f"Confusion matrix: {result['confusion_matrix']}")
    print()

Count unigram NB 80/20
Train size: 9599
Test size: 2401
Vocabulary size: 4840
Training accuracy: 0.9952
Testing accuracy: 0.9829
Confusion matrix: {'negatif': {'negatif': 964, 'netral': 3, 'positif': 1}, 'netral': {'negatif': 10, 'netral': 788, 'positif': 1}, 'positif': {'negatif': 10, 'netral': 16, 'positif': 608}}

TF-IDF unigram+bigram NB 80/20
Train size: 9599
Test size: 2401
Vocabulary size: 16036
Training accuracy: 0.9532
Testing accuracy: 0.8776
Confusion matrix: {'negatif': {'negatif': 949, 'netral': 18, 'positif': 1}, 'netral': {'negatif': 195, 'netral': 593, 'positif': 11}, 'positif': {'negatif': 36, 'netral': 33, 'positif': 565}}

TF-IDF unigram+bigram NB 70/30
Train size: 8399
Test size: 3601
Vocabulary size: 16720
Training accuracy: 0.9386
Testing accuracy: 0.8742
Confusion matrix: {'negatif': {'negatif': 1426, 'netral': 24, 'positif': 2}, 'netral': {'negatif': 307, 'netral': 888, 'positif': 3}, 'positif': {'negatif': 73, 'netral': 44, 'positif': 834}}


## 3. Ringkasan Hasil

In [3]:
best_result = max(results, key=lambda item: item['test_accuracy'])
best_experiment = best_result['experiment']

print(f'Best experiment: {best_experiment.name}')
print(f"Best training accuracy: {best_result['train_accuracy']:.4f}")
print(f"Best testing accuracy: {best_result['test_accuracy']:.4f}")

Best experiment: Count unigram NB 80/20
Best training accuracy: 0.9952
Best testing accuracy: 0.9829


## 4. Inference

In [4]:
from train_sentiment import predict_sentiment

sample_text = 'Aplikasinya mudah dipakai, transaksi cepat, dan sangat membantu.'
prediction = predict_sentiment(
    sample_text,
    best_result['vectorizer'],
    best_result['model'],
)

print(f'Teks: {sample_text}')
print(f'Prediksi kelas: {prediction}')

Teks: Aplikasinya mudah dipakai, transaksi cepat, dan sangat membantu.
Prediksi kelas: positif
